In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f'{catalog_name}.{gold_schema}.dim_drivers'

**Step 1 - Read source tables**

- `silver.constructors`
- `gold.ref_nationality_region`

In [0]:
drivers_df = spark.read.table(f"{catalog_name}.{silver_schema}.drivers")
ref_nationality_region_df = spark.read.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

**Step 2 - Join `drivers` with `nationality_region_df` using `nationality`**

In [0]:
dim_drivers_df = (
            drivers_df
                .join(ref_nationality_region_df, 
                    drivers_df.nationality == ref_nationality_region_df.nationality,
                    "left")
                .select(
                    drivers_df.driver_id,
                    drivers_df.driver_name,
                    drivers_df.date_of_birth,
                    drivers_df.nationality,
                    ref_nationality_region_df.region.alias('nationality_region')
                )
)

**Step 3 - Write the transformed data to the `gold` `dim_drivers` table**

In [0]:
(
    dim_drivers_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(target_table)
)

In [0]:
%sql
select * from formula1.gold.dim_drivers